# Notebook 3a — Description statistique de la cohorte

**Objectif :** Statistiques descriptives completes : demographie, tabac, mutations, exposition aux polluants. Genere les figures pour le rapport.

---

## Donnees en entree

- `../data/1_patients/patients_geocoded_clean_idf_2018_2023.csv` — cohorte geocodee IDF 2018-2023
- `../data/0_brut/socio_eco/classification_histologique_revueCB.xlsx` — classification histologique
- `../data/2_exposition/patients_exposition_metrique_after_2018.csv` — metriques d'exposition (optionnel)

## Donnees en sortie

- `../figures/descriptives/` — figures demographiques, mutationnelles, tabagiques
- `../figures/descriptives/PM25/`, `NO2/`, `O3/`, `PM10/` — distributions par polluant
- `../data/1_patients/patients_geocoded_clean_idf_2018_2023.csv` — cohorte filtree (mise a jour)

---

> **RGPD** : Ce notebook traite des donnees de sante pseudonymisees.
> Les fichiers `data/4_confidentiel/` ne doivent jamais etre versionnés sur git.
> **Chemin racine :** `h:/PFE Loice/Notebooks/Loice_Canc-air_2025/loice_pneumodetect/`

# Analyse Descriptive : Cancer du Poumon et Qualité de l'Air

**Question de recherche** : La qualité de l'air est-elle responsable du développement du cancer du poumon chez les non-fumeurs à long terme ?

**Période d'étude** : 2017-2019  
**Polluants analysés** : NO2, O3, PM10, PM2.5

---

##  1. Imports et Configuration

In [ ]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration graphique
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✓ Imports et configuration terminés")

## 🔧 2. Fonctions de Chargement et Préparation des Données

In [ ]:
def charger_donnees(fichier_patients, fichier_air=None):
    """
    Charge les données patients et qualité de l'air
    
    Parameters:
    -----------
    fichier_patients : str
        Chemin vers le fichier CSV des patients
    fichier_air : str, optional
        Chemin vers le fichier de qualité de l'air (si séparé)
    
    Returns:
    --------
    DataFrame
    """
    print("📥 Chargement des données...")
    df = pd.read_csv(fichier_patients,sep =';')
    print(f"   - Patients chargés : {len(df)} lignes, {len(df.columns)} colonnes")
    print(df.columns.to_list())
    
    # Si les données air sont dans un fichier séparé
    if fichier_air:
        df_air = pd.read_csv(fichier_air)
        print(f"   - Données air chargées : {len(df_air)} lignes")
        df = df.merge(df_air, on='record_id', how='left')
        print(f"   - Fusion réalisée : {len(df)} lignes finales")
    
    return df

In [ ]:
# =============================================================================================
# CLASSIFICATION HISTOLOGIQUE — strictement basée sur le fichier de classification histologique
# =============================================================================================

# ── Chargement du fichier de référence ───────────────────────────────────────
df_ref = pd.read_excel(
    r'../data/0_brut/socio_eco/classification_histologique_revueCB.xlsx',
    header=None
)

# Extraire les lignes utiles (à partir de la ligne 3 = index 2)
df_ref = df_ref.iloc[3:, :2].copy()
df_ref.columns = ['type_histologique', 'categorie']
df_ref = df_ref.dropna(subset=['type_histologique', 'categorie'])

# Normalisation pour matching robuste (minuscules, strip)
df_ref['type_histo_norm'] = df_ref['type_histologique'].astype(str).str.strip().str.lower()
df_ref['categorie']       = df_ref['categorie'].astype(str).str.strip()

# Dictionnaire de mapping exact
MAPPING_HISTO = dict(zip(df_ref['type_histo_norm'], df_ref['categorie']))

print(f"Mapping chargé : {len(MAPPING_HISTO)} entrées")
print("\nAperçu du mapping :")
for k, v in list(MAPPING_HISTO.items())[:10]:
    print(f"  '{k}' → '{v}'")


# =============================================================================
# FONCTION DE CLASSIFICATION
# =============================================================================

def classifier_histologie(histo):
    """
    Classification stricte basée sur le fichier de référence CB.
    Matching exact après normalisation (minuscules + strip + accents).
    Si la valeur n'est pas dans le fichier → 'Non classé'
    """
    if pd.isna(histo) or str(histo).strip() == '':
        return 'Non classé'

    # Normalisation identique au mapping
    h = (str(histo).strip().lower()
         .replace('é', 'e').replace('è', 'e').replace('ê', 'e')
         .replace('à', 'a').replace('â', 'a')
         .replace('ï', 'i').replace('î', 'i')
         .replace('ô', 'o').replace('ù', 'u').replace('ç', 'c'))

    # Normaliser aussi les clés du mapping
    for cle, categorie in MAPPING_HISTO.items():
        cle_norm = (cle.replace('é', 'e').replace('è', 'e').replace('ê', 'e')
                       .replace('à', 'a').replace('â', 'a')
                       .replace('ï', 'i').replace('î', 'i')
                       .replace('ô', 'o').replace('ù', 'u').replace('ç', 'c'))
        if h == cle_norm:
            return categorie

    return 'Non classé'




In [ ]:
# À MODIFIER : Remplacez par le chemin de votre fichier
fichier = r'../data/patients_geocoded_idf.csv'
# fichier = r'../data/2_exposition/patients_exposition_metrique_after_2018.csv'


# Charger les données
df = charger_donnees(fichier)
print('total patients en IDF :', len(df))

df['date_diagnostic'] = pd.to_datetime(df['date_diagnostic'], errors='coerce')
df = df[(df['date_diagnostic'].dt.year >= 2018) & (df['date_diagnostic'].dt.year <= 2023)]  # Filtrer pour les diagnostics entre 2018 et 2023
print('patients avec diagnostic entre 2018 et 2023 :', len(df))


df = df[~df['paquet_annee'].isna()]  # Garder uniquement les patients avec données de tabagisme
print('patients avec données de tabagisme :', len(df))
print('patients avec données de tabagisme et diagnostic entre 2018 et 2023 :', len(df))


# ── Remplacer COL_HISTO par le nom exact de votre colonne ────────────────────
COL_HISTO = 'type_histologique'   # ← à adapter si nécessaire

df['histologie_groupe'] = df[COL_HISTO].apply(classifier_histologie)

# ── Résultats ─────────────────────────────────────────────────────────────────
print("Distribution histologie_groupe :")
print(df['histologie_groupe'].value_counts())

# Valeurs non trouvées dans le fichier de référence
non_classes = df.loc[df['histologie_groupe'] == 'Non classé', COL_HISTO]
print(f"\n⚠️  Valeurs 'Non classé' : {len(non_classes)}")
if len(non_classes) > 0:
    print("Valeurs à ajouter dans le fichier de référence :")
    print(non_classes.value_counts())

df.to_csv(r'../data/1_patients/patients_geocoded_clean_idf_2018_2023.csv', index=False)


In [ ]:
df.type_histologique.unique()
df.histologie_groupe.unique()

In [ ]:
# nombre total de patients uniques dans tout le dataset
total_patients = df["pseudo_provisoire"].nunique()

# agrégation par département
resultat = (
    df.groupby("CODE_DEPT")["pseudo_provisoire"]
      .nunique()
      .reset_index(name="nb_patients")
)

# calcul du pourcentage
resultat["pourcentage"] = resultat["nb_patients"] / total_patients * 100

# option : arrondir
resultat["pourcentage"] = resultat["pourcentage"].round(2)

resultat




In [ ]:
plt.figure()

df_plot = resultat.sort_values("pourcentage", ascending=False)

ax = sns.barplot(
    data=df_plot,
    x="CODE_DEPT",
    y="pourcentage",
    palette="viridis",
    order=df_plot["CODE_DEPT"]
)

# ajouter les valeurs au-dessus des barres
for i, v in enumerate(df_plot["pourcentage"]):
    ax.text(i, v + 0.2, f"{v:.0f}%", ha="center", va="bottom")

plt.title("Geographical distribution per department (%)")
plt.xlabel("Départment")
plt.ylabel("Percentage")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calcul des statistiques
stats_sexe = df['sexe'].value_counts().reset_index()
stats_sexe.columns = ['Sexe', 'Nombre']
stats_sexe['Pourcentage (%)'] = (stats_sexe['Nombre'] / len(df) * 100).round(1)

print("--- Répartition par Sexe ---")
print(stats_sexe.to_string(index=False))

# 2. Création du graphique
plt.figure(figsize=(8, 6))
sns.set_style("whitegrid")

# Création des barres
ax = sns.countplot(data=df, x='sexe', palette='viridis')

# Ajout des étiquettes (nombres et %) sur le graphique
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    count = int(p.get_height())
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(f'{count}\n({percentage})', (x, y), ha='center', va='bottom', fontweight='bold')

plt.title('Répartition des patients par Sexe', fontsize=14)
plt.xlabel('Sexe', fontsize=12)
plt.ylabel('Nombre de patients', fontsize=12)
plt.ylim(0, max(stats_sexe['Nombre']) * 1.15) # Marge en haut pour les étiquettes

plt.show()

In [ ]:
# ======================================================================
# 1. TABLE DE CONTINGENCE
# ======================================================================
table = pd.crosstab(df["statut_tabagique"], df["sexe"])

# trier selon le total décroissant du statut tabagique
table = table.loc[table.sum(axis=1).sort_values(ascending=False).index]

table = table.rename(index={
    "non fumeur": "never smoker",
    "fumeur": "smoker / former smoker",
    "non disponible": "non available"
})

# ======================================================================
# 2. GRAPHIQUE STACKED AVEC COULEURS
# ======================================================================
sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(11, 7))

# définir les couleurs fixes
color_map = {
    "masculin": "#1f77b4",  # bleu
    "feminin": "#e6cbcf"    # rose pâle
}

# s'assurer que les colonnes sont dans le bon ordre
cols = [col for col in ["masculin", "feminin"] if col in table.columns]

table[cols].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    edgecolor="black",
    linewidth=1.5,
    color=[color_map[col] for col in cols]
)

# ======================================================================
# 3. AJOUT DES POURCENTAGES (ignorer "Non disponible")
# ======================================================================
totals = table.sum(axis=1)
min_height = 10  # seuil pour texte à l'intérieur

for i, statut in enumerate(table.index):
    if "non available" in statut.lower():
        continue  # on ignore cette catégorie

    cumulative = 0
    for sexe in cols:
        val = table.loc[statut, sexe]
        if val > 0:
            pct = val / totals[statut] * 100
            
            # texte à l'intérieur si barre suffisamment grande
            if val >= min_height:
                y_pos = cumulative + val / 2
                color = "black"
            else:
                y_pos = cumulative + val + 1  # au-dessus
                color = "black"
            
            ax.text(
                i,
                y_pos,
                f"{pct:.1f}%",
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold",
                color=color
            )
            cumulative += val

# ======================================================================
# 4. MISE EN FORME
# ======================================================================
ax.set_title(
    "Sex Distribution by Smoking Status",
    fontsize=20,
    fontweight="bold",
    pad=20
)

ax.set_ylabel("Number of Patients", fontsize=14, fontweight="bold")
ax.set_xlabel("")
plt.xticks(rotation=0)

sns.despine(left=True)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
table

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================================
# 1. PRÉPARATION & TRI DÉCROISSANT
# ============================================================================
# On s'assure que les données sont nettoyées (logique précédente)
counts = df['statut_tabagique'].value_counts().sort_values(ascending=False)
total = len(df)

# Mapping des couleurs "adéquates"
# Vert émeraude (santé), Rouge corail (risque), Gris doux (inconnu)
palette_couleurs = {
    'non fumeur': '#27ae60', 
    'fumeur': '#e74c3c', 
    'non disponible': '#95a5a6'
}

# Mapping des labels pour l'export anglais
labels_en = {
    'non fumeur': 'Never-Smoker', 
    'fumeur': 'Smoker / Former', 
    'non disponible': 'Not Available'
}

# Création de la liste de couleurs dans l'ordre décroissant des barres
colors_ordered = [palette_couleurs.get(x, '#95a5a6') for x in counts.index]

# ============================================================================
# 2. VISUALISATION HAUTE QUALITÉ
# ============================================================================
sns.set_theme(style="white") # Fond blanc pur pour le contraste
fig, ax = plt.subplots(figsize=(11, 7))

# Création du barplot
# On utilise alpha=0.8 pour une couleur plus "mate" et pro
bars = sns.barplot(
    x=counts.index, 
    y=counts.values, 
    palette=colors_ordered,
    edgecolor='white',
    linewidth=2,
    alpha=0.85,
    ax=ax
)

# Ajout des annotations (Effectifs et Pourcentages)
for i, val in enumerate(counts.values):
    pct = (val / total) * 100
    ax.text(
        i, val + (max(counts.values) * 0.02), # Position juste au-dessus
        f'n = {int(val)}\n({pct:.1f}%)', 
        ha='center', va='bottom', 
        fontsize=14, fontweight='bold', color='#2c3e50'
    )

# --- PERSONNALISATION DES AXES ---
ax.set_title('Cohort Distribution by Smoking Status', fontsize=20, fontweight='bold', pad=30, color='#2c3e50')
ax.set_ylabel('Number of Patients', fontsize=14, fontweight='bold', color='#34495e')
ax.set_xlabel('')

# Remplacement des labels par l'anglais et mise en forme
ax.set_xticklabels([labels_en.get(x, x) for x in counts.index], fontsize=13, fontweight='bold')

# Nettoyage du graphique (Style épuré)
sns.despine(left=True) # Supprime la bordure gauche
ax.yaxis.grid(True, linestyle='--', alpha=0.4) # Grille horizontale discrète
ax.set_ylim(0, max(counts.values) * 1.2) # Marge pour les étiquettes

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# ============================================================================
# 1. PRÉPARATION DES DONNÉES
# ============================================================================
DRIVERS_LIST = ['EGFR', 'ALK', 'ROS1', 'ERBB2', 'RET', 'NTRK', 'MET']

def get_detailed_status(row):
    val = str(row['mutation']).upper().strip()
    if val in ['NO MUTATION', 'NAN', 'NONE']:
        return 'Wild-Type'
    if any(driver in val for driver in DRIVERS_LIST):
        return 'Driver Mutation'
    else:
        return 'Other Mutation Only'

df['detailed_status'] = df.apply(get_detailed_status, axis=1)

counts = df['detailed_status'].value_counts()
n_wt = counts.get('Wild-Type', 0)
n_driver = counts.get('Driver Mutation', 0)
n_others = counts.get('Other Mutation Only', 0)
n_mutated = n_driver + n_others
total = len(df)

# ============================================================================
# 2. CRÉATION DU GRAPHIQUE (DÉGRADÉ DE ROUGES)
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 10))
size = 0.35 

# --- ANNEAU INTÉRIEUR (Détails) ---
vals_inner = [n_driver, n_others, n_wt]
# Dégradé : Rouge vif pour Drivers, Corail pour Others, Gris pour WT
colors_inner = ['#E74C3C', '#F1948A', '#BDC3C7'] 
labels_in = ['DRIVERS', 'OTHERS', 'NO MUTATION']

wedges_in, _ = ax.pie(vals_inner, radius=1-size, colors=colors_inner,
                       wedgeprops=dict(width=size, edgecolor='white', linewidth=3))

for i, p in enumerate(wedges_in):
    ang = (p.theta2 - p.theta1)/2. + p.theta1
    x, y = np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))
    pct = (vals_inner[i]/total)*100
    if pct > 2:
        # Texte en blanc pour les rouges, en gris foncé pour le gris clair
        color_t = 'white' if i < 2 else '#2C3E50'
        ax.text(x*0.48, y*0.48, f"{labels_in[i]}\nn={vals_inner[i]}\n{pct:.1f}%", 
                ha='center', va='center', color=color_t, fontweight='bold', fontsize=9)

# --- ANNEAU EXTÉRIEUR (Synthèse) ---
vals_outer = [n_mutated, n_wt]
# Rouge foncé pour Mutated, Gris moyen pour WT
colors_outer = ['#922B21', '#7F8C8D'] 
labels_out = ['MUTATED', 'NO MUTATION']

wedges_out, _ = ax.pie(vals_outer, radius=1, colors=colors_outer,
                        wedgeprops=dict(width=size, edgecolor='white', linewidth=3))

for i, p in enumerate(wedges_out):
    ang = (p.theta2 - p.theta1)/2. + p.theta1
    x, y = np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))
    pct = (vals_outer[i]/total)*100
    ax.text(x*0.82, y*0.82, f"{labels_out[i]}\nn={vals_outer[i]}\n({pct:.1f}%)", 
            ha='center', va='center', color='white', fontweight='bold', fontsize=10)

# --- CENTRE & FINITIONS ---
ax.text(0, 0, f'TOTAL\nN = {total}', ha='center', va='center', 
        fontsize=15, fontweight='bold', color='#2C3E50')

ax.set_title('Genomic Profile', fontsize=18, fontweight='bold', pad=30)

legend_elements = [
    Line2D([0], [0], marker='s', color='w', label='Drivers ', markerfacecolor='#E74C3C', markersize=12),
    Line2D([0], [0], marker='s', color='w', label='Others', markerfacecolor='#F1948A', markersize=12),
    Line2D([0], [0], marker='s', color='w', label='No Mutation ', markerfacecolor='#BDC3C7', markersize=12)
]
ax.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=3, frameon=False)

plt.show()

# Nettoyage
df.drop(columns=['detailed_status'], inplace=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import re  # Pour nettoyer le nom de fichier

# ============================================================================
# 1. PRÉPARATION DES DONNÉES
# ============================================================================
# DRIVERS_LIST = ['EGFR', 'ALK', 'ROS1', 'ERBB2', 'MET', 'RET', 'NTRK', 'KRAS', 'BRAF']

def get_detailed_status(val):
    val = str(val).upper().strip()
    if val in ['NO MUTATION', 'NAN', 'NONE']:
        return 'No Mutation'
    if any(driver in val for driver in DRIVERS_LIST):
        return 'Driver'
    else:
        return 'Other'

df['status'] = df['mutation'].apply(get_detailed_status)
df['tabac_group'] = df['statut_tabagique'].fillna('non disponible')

# Listes pour la boucle
tabac_types = ['fumeur', 'non fumeur', 'non disponible']
display_titles = ['SMOKERS / FORMER', 'NEVER-SMOKERS', 'NOT AVAILABLE']

# ============================================================================
# 2. GÉNÉRATION ET SAUVEGARDE (SÉPARÉE)
# ============================================================================
for i, tabac in enumerate(tabac_types):
    subset = df[df['tabac_group'] == tabac]
    n_group = len(subset)
    if n_group == 0: continue

    counts = subset['status'].value_counts()
    n_driver = counts.get('Driver', 0)
    n_others = counts.get('Other', 0)
    n_wt = counts.get('No Mutation', 0)
    n_mutated = n_driver + n_others

    fig, ax = plt.subplots(figsize=(14, 14)) # Encore plus grand
    size = 0.35

    # --- ANNEAU INTÉRIEUR ---
    vals_in = [n_driver, n_others, n_wt]
    wedges_in, _ = ax.pie(vals_in, radius=1-size, colors=['#E74C3C', '#F1948A', '#BDC3C7'],
                          wedgeprops=dict(width=size, edgecolor='white', linewidth=5))

    for j, p in enumerate(wedges_in):
        ang = (p.theta2 - p.theta1)/2. + p.theta1
        x, y = np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))
        pct = (vals_in[j]/n_group)*100
        if pct > 2:
            color_t = 'white' if j < 2 else '#2C3E50'
            ax.text(x*0.48, y*0.48, f"{vals_in[j]}\n{pct:.1f}%", 
                    ha='center', va='center', color=color_t, fontweight='bold', fontsize=20)

    # --- ANNEAU EXTÉRIEUR ---
    vals_out = [n_mutated, n_wt]
    wedges_out, _ = ax.pie(vals_out, radius=1, colors=['#922B21', '#7F8C8D'],
                           wedgeprops=dict(width=size, edgecolor='white', linewidth=5))

    for j, p in enumerate(wedges_out):
        ang = (p.theta2 - p.theta1)/2. + p.theta1
        x, y = np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))
        pct = (vals_out[j]/n_group)*100
        if pct > 0:
            ax.text(x*0.84, y*0.84, f"{pct:.1f}%", 
                    ha='center', va='center', color='white', fontweight='bold', fontsize=22)

    # --- CENTRE & TITRE ---
    ax.text(0, 0, f"TOTAL\nN = {n_group}", ha='center', va='center', 
            fontsize=26, fontweight='bold', color='#2C3E50')
    
    ax.set_title(f"{display_titles[i]} GROUP", fontsize=32, fontweight='black', pad=60)

    # Légende
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', label='Drivers', markerfacecolor='#E74C3C', markersize=25),
        Line2D([0], [0], marker='s', color='w', label='Other Mutations', markerfacecolor='#F1948A', markersize=25),
        Line2D([0], [0], marker='s', color='w', label='No Mutation ', markerfacecolor='#BDC3C7', markersize=25)
    ]
    ax.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.15), 
              ncol=1, frameon=False, fontsize=20)

    # --- SAUVEGARDE SÉCURISÉE ---
    # Nettoyer le nom de fichier (remplace tout ce qui n'est pas lettre/chiffre par un underscore)
    file_label = re.sub(r'[^a-zA-Z0-9]', '_', display_titles[i])
    filename = f"Graphique_{file_label}.png"
    
    plt.savefig(filename, dpi=600, bbox_inches='tight', facecolor='white')
    print(f"Succès : Graphique enregistré sous '{filename}'")
    
    plt.show()

# Nettoyage
df.drop(columns=['status', 'tabac_group'], inplace=True)

In [ ]:

import re

# ============================================================================
# 1. CALCUL DU NOMBRE DE MUTATIONS PAR PATIENT
# ============================================================================
def count_mutations(val):
    val = str(val).upper().strip()
    # Si 'NO MUTATION' ou vide -> 0
    if val in ['NO MUTATION', 'NAN', 'NONE', '']:
        return 0
    
    # On sépare par les délimiteurs classiques (virgule, slash, espace)
    # On filtre les éléments vides pour ne pas surcompter
    mut_list = [m for m in re.split(r'[,/\s]+', val) if m.strip()]
    return len(mut_list)

df['nb_mutations'] = df['mutation'].apply(count_mutations)

# Préparation des données pour le graphique
count_data = df['nb_mutations'].value_counts().sort_index()
total_patients = len(df)

# ============================================================================
# 2. VISUALISATION HAUTE RÉSOLUTION (600 DPI)
# ============================================================================
plt.figure(figsize=(12, 10))
colors = sns.color_palette("Reds", len(count_data)) # Dégradé de rouge cohérent

bars = plt.bar(count_data.index.astype(str), count_data.values, 
                color=colors, edgecolor='#922B21', linewidth=2)

# Ajout des labels n et % au-dessus de chaque barre
for bar in bars:
    height = bar.get_height()
    percentage = (height / total_patients) * 100
    plt.text(bar.get_x() + bar.get_width()/2., height + (total_patients*0.01),
             f'n={int(height)}\n({percentage:.1f}%)',
             ha='center', va='bottom', fontsize=14, fontweight='bold', color='#2C3E50')

# --- ESTHÉTIQUE ---
plt.title('DISTRIBUTION OF MUTATIONS\n(Number of Mutations per Patient)', 
          fontsize=22, fontweight='black', pad=40)
plt.xlabel('Number of Co-occurring Mutations', fontsize=16, fontweight='bold')
plt.ylabel('Number of Patients', fontsize=16, fontweight='bold')

plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=12)

# Suppression des bordures inutiles
sns.despine()

# -----------------------------------------------------------------------
# SAUVEGARDE 600 DPI
# -----------------------------------------------------------------------
filename = "Distribution_Nombre_Mutations_600DPI.png"
plt.savefig(filename, dpi=600, bbox_inches='tight', facecolor='white')
print(f"Graphique enregistré avec succès : {filename}")

plt.show()

# Nettoyage
# df.drop(columns=['nb_mutations'], inplace=True) # Optionnel

In [ ]:
df.head(5)

In [ ]:
pd.crosstab(df["histologie_groupe"], df["statut_tabagique"])

In [ ]:
df.type_histologique.unique()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re

# ============================================================================
# 1. PRÉPARATION ET COMPTAGE (CLASSEMENT DÉCROISSANT)
# ============================================================================
# DRIVERS_LIST = ['EGFR', 'ALK', 'ROS1', 'ERBB2', 'MET', 'RET', 'NTRK', 'KRAS', 'BRAF']

# Dictionnaires pour stocker les résultats
n_no_mutation = 0
n_other_mutation = 0
driver_counts = {gene: 0 for gene in DRIVERS_LIST}

for val in df['mutation']:
    val = str(val).upper().strip()
    
    # 1. Cas : Aucune mutation
    if val in ['NO MUTATION', 'NAN', 'NONE', '']:
        n_no_mutation += 1
        continue
    
    # Extraction des gènes
    genes = [g.strip() for g in re.split(r'[,/\s]+', val) if g.strip()]
    
    # 2. Cas : Vérifier la présence de Drivers
    found_drivers = [g for g in genes if g in DRIVERS_LIST]
    
    if found_drivers:
        # On compte chaque driver présent
        for d in found_drivers:
            driver_counts[d] += 1
    else:
        # 3. Cas : Mutation identifiée mais aucun driver de la liste
        n_other_mutation += 1

# Construction du DataFrame de synthèse
data = []
data.append({'Label': 'NO MUTATION', 'Count': n_no_mutation, 'ColorGroup': 'None'})
data.append({'Label': 'OTHER MUTATION', 'Count': n_other_mutation, 'ColorGroup': 'Other'})

for gene, count in driver_counts.items():
    if count > 0:
        data.append({'Label': gene, 'Count': count, 'ColorGroup': 'Driver'})

# --- TRI PAR ORDRE DÉCROISSANT ---
df_final = pd.DataFrame(data).sort_values(by='Count', ascending=False)

# ============================================================================
# 2. VISUALISATION HAUTE RÉSOLUTION (600 DPI)
# ============================================================================
plt.figure(figsize=(16, 12))

# Application des couleurs demandées
# Rouge pour Drivers, Bleu pour Autres, Gris pour Aucune
color_map = {'Driver': '#E74C3C', 'Other': '#3498DB', 'None': '#BDC3C7'}
colors = [color_map[t] for t in df_final['ColorGroup']]

bars = plt.barh(df_final['Label'], df_final['Count'], color=colors, edgecolor='white', linewidth=2)

# Ajout des étiquettes de données (n et %)
total_pts = len(df)
for bar in bars:
    width = bar.get_width()
    plt.text(width + (total_pts * 0.005), bar.get_y() + bar.get_height()/2,
             f'n={int(width)} ({(width/total_pts)*100:.1f}%)',
             va='center', fontsize=18, fontweight='bold', color='#2C3E50')

# --- ESTHÉTIQUE ---
plt.title('GENOMIC LANDSCAPE RANKING', 
          fontsize=30, fontweight='black', pad=50)
plt.xlabel('Number of Patients', fontsize=20, fontweight='bold')
plt.yticks(fontsize=20, fontweight='bold')

# Inverser l'axe Y pour avoir le plus grand en haut
plt.gca().invert_yaxis()

sns.despine()

# -----------------------------------------------------------------------
# SAUVEGARDE 600 DPI
# -----------------------------------------------------------------------
filename = "Ranking_Global_Mutations_600DPI.png"
plt.savefig(filename, dpi=600, bbox_inches='tight', facecolor='white')
print(f"Graphique final enregistré : {filename}")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re

# ============================================================================
# 1. PRÉPARATION DES DONNÉES FILTRÉES
# ============================================================================
# DRIVERS_LIST = ['EGFR', 'KRAS', 'ALK', 'BRAF', 'MET', 'ROS1', 'ERBB2', 'RET', 'NTRK']

expanded_data = []

for _, row in df.iterrows():
    # Gestion sécurisée du statut tabagique
    tabac_raw = row['statut_tabagique']
    tabac = str(tabac_raw) if pd.notna(tabac_raw) else 'non disponible'
    
    val = str(row['mutation']).upper().strip()
    
    # Cas 1 : Aucune mutation
    if val in ['NO MUTATION', 'NAN', 'NONE', '']:
        expanded_data.append({'Label': 'NO MUTATION', 'Tabac': tabac})
    else:
        # Séparation des gènes (gestion des multi-mutations)
        genes = [g.strip() for g in re.split(r'[,/\s]+', val) if g.strip()]
        
        for g in genes:
            if g in DRIVERS_LIST:
                # On garde le nom spécifique du Driver
                expanded_data.append({'Label': g, 'Tabac': tabac})
            else:
                # Tout le reste devient 'OTHER'
                expanded_data.append({'Label': 'OTHER', 'Tabac': tabac})

df_expanded = pd.DataFrame(expanded_data)

# Création du tableau croisé (Gène en index, Tabac en colonnes)
ct = pd.crosstab(df_expanded['Label'], df_expanded['Tabac'])

# --- TRI DÉCROISSANT ---
ct['Total'] = ct.sum(axis=1)
ct = ct.sort_values(by='Total', ascending=False).drop(columns='Total')

# ============================================================================
# 2. VISUALISATION (600 DPI)
# ============================================================================
# Couleurs conformes à votre image de référence
color_map = {
    'fumeur': '#E74C3C',        # Rouge
    'non fumeur': '#2ECC71',    # Vert
    'non disponible': '#95A5A6' # Gris
}

# Ordre des segments dans les barres
order_tabac = [c for c in ['non fumeur', 'fumeur', 'non disponible'] if c in ct.columns]
ct_plot = ct[order_tabac]

fig, ax = plt.subplots(figsize=(16, 10))

# Tracé de l'histogramme empilé
ct_plot.plot(kind='bar', stacked=True, ax=ax,
             color=[color_map[c] for c in order_tabac], 
             width=0.75, edgecolor='white', linewidth=1)

# Ajout des totaux n=... au-dessus de chaque barre
for i, (label, row_sum) in enumerate(ct_plot.iterrows()):
    total = row_sum.sum()
    ax.text(i, total + (ct_plot.values.max() * 0.02), f'n = {int(total)}', 
            ha='center', va='bottom', fontsize=14, fontweight='bold', color='#2C3E50')

# --- ESTHÉTIQUE FINALE ---
ax.set_title('POSITIVE MUTATIONS BY GENE AND SMOKING STATUS', 
             fontsize=26, fontweight='black', pad=40)
ax.set_ylabel('Number of Positive Mutations', fontsize=18, fontweight='bold')
ax.set_xlabel('Gene / Mutation Category', fontsize=18, fontweight='bold')

plt.xticks(rotation=45, ha='right', fontsize=15, fontweight='bold')
plt.yticks(fontsize=13)

# Légende
handles, labels = ax.get_legend_handles_labels()
labels_clean = [l.replace('fumeur', 'Smoker').replace('non disponible', 'Not available').title() for l in labels]
ax.legend(handles, labels_clean, title="Smoking Status", title_fontsize='15', 
          fontsize='13', frameon=False, bbox_to_anchor=(1, 1))

plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()

# -----------------------------------------------------------------------
# SAUVEGARDE HAUTE RÉSOLUTION
# -----------------------------------------------------------------------
filename = "Synthese_Mutations_Tabac_Principales_600DPI.png"
plt.savefig(filename, dpi=600, bbox_inches='tight')
print(f"Fichier enregistré : {filename}")

plt.show()

In [ ]:
ct

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. PRÉPARATION
label_mapping = {
    'Non exposé': 'Never Smoker',
    'Faible (1-10)': 'Light (1-10)',
    'Modérée (10-20)': 'Moderate (10-20)',
    'Forte (20-40)': 'Heavy (20-40)',
    'Très forte (>40)': 'Very Heavy (>40)'
}
df['exposition_tabagique_en'] = df['exposition_tabagique'].fillna('non defini').map(label_mapping)
# mask = (
#     df['exposition_tabagique'].isna()
#     & (df['statut_tabagique'] == 'non disponible')
# )

# df.loc[mask, 'exposition_tabagique_en'] = (
#     'non defini'
# )

# df['exposition_tabagique_en'] = (
#     df['exposition_tabagique_en']
#     .map(label_mapping)
#     .fillna(df['exposition_tabagique_en'])
# )

english_order = ['Never Smoker', 'Light (1-10)', 'Moderate (10-20)', 'Heavy (20-40)', 'Very Heavy (>40)']
expo_counts = df['exposition_tabagique_en'].value_counts().reindex(english_order).fillna(0)

# 2. VISUALISATION
plt.figure(figsize=(14, 10))

# Utilisation de la palette 'rocket' pour un dégradé intense
gradient_palette = sns.color_palette("OrRd", n_colors=4) # _r pour inverser et avoir le sombre à la fin
colors = ['#27ae60'] + list(gradient_palette)  

bars = plt.bar(expo_counts.index, expo_counts.values, color=colors, edgecolor='none')

# Labels n et %
total_pts = len(df)
for bar in bars:
    height = bar.get_height()
    pct = (height / total_pts) * 100
    plt.text(bar.get_x() + bar.get_width()/2., height + (total_pts * 0.01),
             f'n={int(height)}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=12, fontweight='bold', color='#2F3640')

# Esthétique
plt.title(" DISTRIBUTION OF SMOKING EXPOSURE", fontsize=24, fontweight='black', pad=40)
plt.ylabel('Number of Patients', fontsize=16, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')

sns.despine(left=True)
plt.grid(axis='y', linestyle='-', alpha=0.1)

plt.savefig("Smoking_Exposure_Rocket_600DPI.png", dpi=600, bbox_inches='tight')
# df = df.drop(columns=['exposition_tabagique_en'])
plt.show()

In [ ]:
# =============================================================================
# DISTRIBUTION OF SMOKING EXPOSURE — basé sur paquet_annee
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def plot_smoking_distribution(df, save_path=None):
    """
    Barplot de distribution de l'exposition tabagique basé sur paquet_annee.
    
    Catégories :
    - Never Smoker : PA = 0 ou NaN avec statut non-fumeur
    - Light (≤5 PA) : 0 < PA ≤ 5
    - Moderate (5-20 PA) : 5 < PA ≤ 20
    - Heavy (>20 PA) : PA > 20
    """
    
    df = df.copy()
    
    # Convertir paquet_annee en numérique
    df['pa'] = pd.to_numeric(df['paquet_annee'], errors='coerce')
    
    # Catégorisation basée sur paquet_annee
    def categoriser_pa(row):
        pa = row['pa']
        statut = str(row.get('statut_tabagique', '')).lower().strip()
        
        # Non-fumeur
        if pd.isna(pa) or pa == 0:
            if statut in ['jamais fumé', 'non-fumeur', 'never smoker', 'jamais', 'non fumeur']:
                return 'Never Smoker'
            elif pd.isna(pa):
                return None  # Exclure les non définis
            else:
                return 'Never Smoker'
        elif pa <= 5:
            return 'Light (≤5 PA)'
        elif pa <= 20:
            return 'Moderate (5-20 PA)'
        else:
            return 'Heavy (>20 PA)'
    
    df['smoking_category'] = df.apply(categoriser_pa, axis=1)
    
    # Ordre des catégories
    order = ['Never Smoker', 'Light (≤5 PA)', 'Moderate (5-20 PA)', 'Heavy (>20 PA)']
    
    # Compter
    counts = df['smoking_category'].value_counts().reindex(order).fillna(0)
    
    # Figure
    plt.figure(figsize=(12, 9))
    
    # Couleurs : vert pour Never Smoker, dégradé orange-rouge pour fumeurs
    gradient_palette = sns.color_palette("OrRd", n_colors=3)
    colors = ['#27ae60'] + list(gradient_palette)
    
    bars = plt.bar(counts.index, counts.values, color=colors, edgecolor='none')
    
    # Labels n et %
    total_pts = df['smoking_category'].notna().sum()
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            pct = (height / total_pts) * 100
            plt.text(bar.get_x() + bar.get_width()/2., height + (total_pts * 0.01),
                     f'n={int(height)}\n({pct:.1f}%)',
                     ha='center', va='bottom', fontsize=13, fontweight='bold', color='#2F3640')
    
    # Esthétique
    plt.title("DISTRIBUTION OF SMOKING EXPOSURE", fontsize=24, fontweight='black', pad=40)
    plt.ylabel('Number of Patients', fontsize=16, fontweight='bold')
    plt.xlabel('')
    plt.xticks(fontsize=13, fontweight='bold')
    plt.yticks(fontsize=11)
    
    sns.despine(left=True)
    plt.grid(axis='y', linestyle='-', alpha=0.1)
    
    # Afficher stats
    print("\n📊 Distribution de l'exposition tabagique :")
    print("=" * 50)
    for cat in order:
        n = int(counts.get(cat, 0))
        pct = n / total_pts * 100 if total_pts > 0 else 0
        print(f"   {cat:<25} : {n:>5} ({pct:>5.1f}%)")
    print("=" * 50)
    print(f"   Total : {total_pts}")
    
    if save_path:
        plt.savefig(save_path, dpi=600, bbox_inches='tight', facecolor='white')
        print(f"\n✅ Figure sauvegardée : {save_path}")
    
    plt.show()
    
    return df


# =============================================================================
# UTILISATION
# =============================================================================

df_expo = plot_smoking_distribution(df, save_path='Smoking_Exposure_600DPI.png')

"""
df_final = pd.read_excel('cohorte.xlsx')
df_final = plot_smoking_distribution(df_final, save_path='Smoking_Exposure_600DPI.png')
"""

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re

# ============================================================================
# 1. PRÉPARATION : "Déplier" et mise en majuscules
# ============================================================================
expanded_data = []
expo_order = ['Never Smoker', 'Light (1-10)', 'Moderate (10-20)', 'Heavy (20-40)', 'Very Heavy (>40)']
main_drivers = ['EGFR','ALK', 'ROS1', 'ERBB2', 'RET', 'MET', 'NTRK']

for _, row in df.iterrows():
    expo = row['exposition_tabagique_en']
    muts_raw = str(row['mutation']).upper().strip()
    
    # Cas "No Mutation"
    if muts_raw in ['NO MUTATION', 'NONE', 'NEGATIVE', 'NAN', '']:
        expanded_data.append({'Gene': 'NO MUTATION', 'Exposure': expo})
        continue

    # Séparation et mise en majuscules systématique
    genes = [g.strip().upper() for g in re.split(r'[,/\s]+', muts_raw) if g.strip()]
    for g in genes:
        if g in main_drivers:
            expanded_data.append({'Gene': g, 'Exposure': expo})
        else:
            expanded_data.append({'Gene': 'OTHER', 'Exposure': expo})

df_cross = pd.DataFrame(expanded_data)

# Création de la table et normalisation
ct = pd.crosstab(df_cross['Gene'], df_cross['Exposure']).reindex(columns=expo_order).fillna(0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

# ============================================================================
# 2. VISUALISATION (600 DPI)
# ============================================================================
plt.figure(figsize=(16, 9))

# Palette d'intensité : Bleu (Never) -> Jaune -> Orange -> Rouge -> Gris (ND)
intensity_colors = [
    '#3498DB', # Never Smoker (Blue)
    '#FEE08B', # Light (Yellow)
    '#FDAE61', # Moderate (Orange)
    '#F46D43', # Heavy (Deep Orange)
    '#D53E4F', # Very Heavy (Red)
]

ax = ct_pct.plot(kind='bar', stacked=True, color=intensity_colors, width=0.8, ax=plt.gca())

# Ajout des % à l'intérieur
for p in ax.patches:
    h = p.get_height()
    if h > 4: 
        ax.text(p.get_x() + p.get_width()/2., p.get_y() + h/2., f'{int(h)}%', 
                ha='center', va='center', color='black' if h < 40 else 'white', 
                fontsize=10, fontweight='bold')

# --- ESTHÉTIQUE FINALE ---
plt.title("SMOKING INTENSITY PROFILE BY GENOMIC STATUS\n(Relative Distribution %)", 
          fontsize=22, fontweight='black', pad=35)
plt.ylabel("Percentage of Patients (%)", fontsize=15, fontweight='bold')
plt.xlabel("GENOMIC ALTERATIONS", fontsize=15, fontweight='bold', labelpad=15)

# Forcer les labels de l'axe X en majuscules (au cas où)
ax.set_xticklabels([t.get_text().upper() for t in ax.get_xticklabels()], 
                   rotation=0, ha='center', fontsize=12, fontweight='bold')

plt.legend(title="Smoking Intensity (PA)", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

sns.despine(left=True)
plt.tight_layout()

# SAUVEGARDE
plt.savefig("Smoking_Mutation_Intensity_Final_600DPI.png", dpi=600, bbox_inches='tight')
plt.show()

### 8.2 Analyses descriptives de la cohorte

In [ ]:
df['sexe'] = df['sexe'].fillna('feminin')
df.sexe.unique()

In [ ]:
# ============================================================================
# HISTPLOT SEXE - Nombre et Pourcentage
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Compter
sexe_counts = df['sexe'].value_counts()

# Couleurs
colors = {'masculin': '#4A90E2', 'feminin': '#E85D75', 'non disponible': '#95A5A6'}
colors_list = [colors.get(x, '#95A5A6') for x in sexe_counts.index]

# Barplot
bars = ax.bar(range(len(sexe_counts)), sexe_counts.values, 
              color=colors_list, edgecolor='white', linewidth=2, alpha=0.85)

# Labels
ax.set_xticks(range(len(sexe_counts)))
ax.set_xticklabels(sexe_counts.index, fontsize=13)
ax.set_ylabel('Nombre de patients', fontsize=14, fontweight='bold')
ax.set_title('Répartition par Sexe', fontsize=16, fontweight='bold', pad=20)

# Ajouter valeurs (n et %)
for bar, val in zip(bars, sexe_counts.values):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(df)*0.02,
            f'n = {val}\n({pct:.1f}%)', 
            ha='center', va='bottom', fontsize=13, fontweight='bold')

# Style
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('repartition_sexe.png', dpi=300, bbox_inches='tight')
plt.show()

# Afficher aussi dans la console
print("\n📊 Répartition par sexe:")
for sexe, count in sexe_counts.items():
    pct = count / len(df) * 100
    print(f"  {sexe}: {count} ({pct:.1f}%)")

In [ ]:
print("\n" + "="*80)
print("🧬 RÉCAPITULATIF DES MUTATIONS")
print("="*80)

# Liste des gènes
genes = ['EGFR', 'ALK', 'ROS1', 'ERBB2', 'RET', 'MET', 'NTRK']

# Compter les mutations POSITIVES principales
total_mutations_principales = 0
patients_avec_mutations_principales = set()

for gene in genes:
    col = f'mutation_{gene}'
    if col in df.columns:
        n_positive = (df[col].str.lower() == 'positive').sum()
        total_mutations_principales += n_positive
        
        patients_positifs = df[df[col].str.lower() == 'positive'].index
        patients_avec_mutations_principales.update(patients_positifs)

nb_patients_mutations_principales = len(patients_avec_mutations_principales)
nb_patients_sans_mutation = (df['mutation'].str.upper() == 'NO MUTATION').sum()
nb_patients_autres_mutations = (df['mutation'].str.upper() == 'AUTRES').sum()

nb_total_patients_mutes = nb_patients_mutations_principales + nb_patients_autres_mutations
total_mutations = total_mutations_principales + nb_patients_autres_mutations

print(f"\n📊 STATISTIQUES GLOBALES:")
print(f"  • Total patients: {len(df)}")
print(f"  • Patients WITH principal mutation(s): {nb_patients_mutations_principales} ({nb_patients_mutations_principales/len(df)*100:.1f}%)")
print(f"  • Patients WITH other mutation(s): {nb_patients_autres_mutations} ({nb_patients_autres_mutations/len(df)*100:.1f}%)")
print(f"  • Patients WITHOUT mutation: {nb_patients_sans_mutation} ({nb_patients_sans_mutation/len(df)*100:.1f}%)")
print(f"  • Total positive mutations: {total_mutations}")
print(f"  • Average mutations per mutated patient: {total_mutations/nb_total_patients_mutes:.2f}" if nb_total_patients_mutes > 0 else "  • Average: N/A")

print("="*80 + "\n")


In [ ]:
df

In [ ]:
# ============================================================================
# HISTOGRAMME - Mutations par gène + Sans mutation
# ============================================================================

fig, ax = plt.subplots(figsize=(16, 8))

# Compter les mutations POSITIVES pour chaque gène
mutation_counts = {}

for gene in genes:
    col = f'mutation_{gene}'
    if col in df.columns:
        n_positive = (df[col].str.lower() == 'positive').sum()
        mutation_counts[gene] = n_positive

# AJOUTER "No mutation"
mutation_counts['NO MUTATION'] = nb_patients_sans_mutation
mutation_counts['AUTRES'] = df[df['mutation_AUTRES']=='positive'].shape[0]

# Trier par ordre décroissant
mutation_counts = dict(sorted(mutation_counts.items(), key=lambda x: x[1], reverse=True))

# Couleurs : rouge pour mutations, gris pour "No mutation"
colors = []
for gene in mutation_counts.keys():
    if gene == 'NO MUTATION':
        colors.append('#95A5A6')  # Gris
    elif gene == 'AUTRES':
        colors.append('#F39C12')  # Orange
    else:
        colors.append('#E74C3C')  # Rouge

# Barplot
bars = ax.bar(mutation_counts.keys(), mutation_counts.values(),
              width=0.7, color=colors, edgecolor='white', linewidth=3, alpha=0.9)

# Ajouter valeurs (n et %)
for bar, (gene, val) in zip(bars, mutation_counts.items()):
    pct = val / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'n = {val}\n({pct:.1f}%)', 
            ha='center', va='bottom', fontsize=14, fontweight='bold')

# Labels en ANGLAIS
ax.set_xlabel('Gene / Mutation Status', fontsize=16, fontweight='bold')
ax.set_ylabel('Number of Patients', fontsize=16, fontweight='bold')
ax.set_title('Number of Patients by Gene Mutation Status', 
             fontsize=18, fontweight='bold', pad=25)

# Rotation des labels X
ax.tick_params(axis='x', labelsize=13, rotation=45)
ax.tick_params(axis='y', labelsize=14)

# Style
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=1.5)

# Ajuster limite Y
ax.set_ylim(0, max(mutation_counts.values()) * 1.2)

plt.tight_layout()
# plt.savefig('mutations_by_gene_with_no_mutation.png', dpi=300, bbox_inches='tight')
plt.show()

# Afficher dans la console
print("\n📊 Distribution:")
for gene, count in mutation_counts.items():
    pct = count / len(df) * 100
    print(f"  {gene:15s}: {count:4d} ({pct:5.1f}%)")
print(f"\nTotal: {len(df)} patients")

In [ ]:
# ============================================================================
# HISTOGRAMME - Nombre de patients diagnostiqués par an
# ============================================================================


# Extraire l'année
df['date_diagnostic'] = pd.to_datetime(df['date_diagnostic'], errors='coerce')
df['annee_diagnostic'] = df['date_diagnostic'].dt.year

# Compter par année
annee_counts = df['annee_diagnostic'].value_counts().sort_index()

# Graphique
fig, ax = plt.subplots(figsize=(18, 12))

bars = ax.bar(annee_counts.index, annee_counts.values, 
              color='#3498DB', edgecolor='white', linewidth=1.5, alpha=1)

# Ajouter valeurs au-dessus des barres
for bar, val in zip(bars, annee_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(annee_counts)*0.01,
            f'{int(val)}', ha='center', va='bottom', fontsize=16, fontweight='bold')

ax.set_xlabel('year of diagnosis', fontsize=30, fontweight='bold')
ax.set_ylabel('Number of patients', fontsize=30, fontweight='bold')
ax.set_title('Number of Patients Diagnosed by Year', fontsize=18, fontweight='bold', pad=20)

# Rotation des années
ax.tick_params(axis='x', rotation=45)

# Style
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
# plt.savefig('diagnostics_par_an.png', dpi=300, bbox_inches='tight')
plt.show()

# Afficher stats
print("\n📊 Diagnosis by year:")
print(f"\nTotal: {annee_counts.sum()} patients")
print(f"Années couvertes: {int(annee_counts.index.min())} - {int(annee_counts.index.max())}")



In [ ]:
# ============================================================================
# HISTOGRAMME - Statut tabagique
# ============================================================================
if 'statut_tabagique' in df.columns:
        df.loc[df['statut_tabagique']=='none', 'statut_tabagique'] = 'non disponible'
        df.loc[df['statut_tabagique']=='non disponible (old)', 'statut_tabagique'] = 'non disponible'


        df['statut_tabagique'] = df['statut_tabagique'].astype(str).str.lower().str.strip()
        df.loc[df['statut_tabagique'] == 'ancien fumeur', 'statut_tabagique'] = 'fumeur'
        
        condition_fumeur = (df['statut_tabagique'].isin(['non disponible', 'none']) & 
                           df['paquet_annee'].notna() & (df['paquet_annee'] > 0))
        df.loc[condition_fumeur, 'statut_tabagique'] = 'fumeur'
        
        condition_non_fumeur = (df['statut_tabagique'].isin(['non disponible', 'none']) & (df['paquet_annee'] == 0))
        df.loc[condition_non_fumeur, 'statut_tabagique'] = 'non fumeur'
        
        df.loc[(df['statut_tabagique'] == 'non fumeur') & 
                     df['paquet_annee'].isna(), 'paquet_annee'] = 0
        
fig, ax = plt.subplots(figsize=(12, 7))  # Plus grand
# Compter
fumeur_counts = df['statut_tabagique'].value_counts()

# Palette de couleurs
palette = {
    'non fumeur': '#2ECC71',       # Vert
    'fumeur': '#E74C3C',           # Rouge
    'non disponible': '#95A5A6'    # Gris
}
colors = [palette.get(x, '#95A5A6') for x in fumeur_counts.index]

# Barplot Seaborn
bars = sns.barplot(x=fumeur_counts.index, y=fumeur_counts.values, 
                   palette=colors, edgecolor='white', linewidth=2, alpha=0.85, ax=ax)

# Ajouter valeurs (n et %) - CORRIGÉ
for i, (statut, val) in enumerate(fumeur_counts.items()):
    pct = val / len(df) * 100
    ax.text(i, val,  # Position au sommet de la barre
            f'n = {val}\n({pct:.1f}%)', 
            ha='center', va='bottom', fontsize=13, fontweight='bold')

# Labels en ANGLAIS
ax.set_xlabel('Smoking Status', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Patients', fontsize=14, fontweight='bold')
ax.set_title('Distribution by Smoking Status', 
             fontsize=16, fontweight='bold', pad=20)

# Renommer les labels sur l'axe X
labels_english = {'non fumeur': 'Non-smoker', 'fumeur': 'Former Smoker/Smoker', 'non disponible': 'Not available'}
ax.set_xticklabels([labels_english.get(x.get_text(), x.get_text()) for x in ax.get_xticklabels()], 
                   fontsize=13)

# Style
sns.despine()
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Ajuster les limites Y pour que les valeurs rentrent
ax.set_ylim(0, max(fumeur_counts) * 1.15)

plt.tight_layout()
# plt.savefig('smoking_status.png', dpi=300, bbox_inches='tight')
plt.show()

# Afficher dans la console (en anglais)
print("\n📊 Smoking Status:")
for statut, count in fumeur_counts.items():
    pct = count / len(df) * 100
    statut_en = labels_english.get(statut, statut)
    print(f"  {statut_en}: {count} ({pct:.1f}%)")
print(f"\nTotal: {fumeur_counts.sum()} patients")

In [ ]:
df_patients_fumeurs = df[df['statut_tabagique'] != 'non fumeur']


In [ ]:
# df.mutation_principale.unique()

In [ ]:
df.columns.to_list()